In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sithukaungset/hairlossdataset/data0330/notbald/patty_cuts_Fresh-cuts-for-black-hair-men-e1464037585603.jpg
/kaggle/input/datasets/sithukaungset/hairlossdataset/data0330/notbald/4-Simple-Hairstyles-For-Kids-With-Short-Hair-2.jpg
/kaggle/input/datasets/sithukaungset/hairlossdataset/data0330/notbald/dreams-about-hair-falling-out-2.jpg
/kaggle/input/datasets/sithukaungset/hairlossdataset/data0330/notbald/Style-Asian-Male-Hair-Step-21-Version-2.jpg
/kaggle/input/datasets/sithukaungset/hairlossdataset/data0330/notbald/menhair0056.jpg
/kaggle/input/datasets/sithukaungset/hairlossdataset/data0330/notbald/frizzy-hair-help.jpg
/kaggle/input/datasets/sithukaungset/hairlossdataset/data0330/notbald/Child-brushing-hair-tips-infacol.jpg
/kaggle/input/datasets/sithukaungset/hairlossdataset/data0330/notbald/blog_15344402631433829142.png
/kaggle/input/datasets/sithukaungset/hairlossdataset/data0330/notbald/kids-cuts-and-color-at-k-bella-in-brighton_3.1000x0.jpg
/kaggle/input/datas

In [2]:
# ===========================
# Imports
# ===========================
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Image transforms.

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [6]:
import os

# Check what datasets are attached to your environment
if os.path.exists('/kaggle/input'):
    print("Available datasets:", os.listdir('/kaggle/input'))
else:
    print("Not running inside a Kaggle /input environment!")


Available datasets: ['datasets']


# DATASET PATH

In [7]:
HAIR_DATASET  = "/kaggle/input/datasets/kavyasreeb/hair-type-dataset/data"

BALD_DATASET  = "/kaggle/input/datasets/sithukaungset/hairlossdataset/data0330"



# Create the dataset

In [8]:
hair_dataset  = datasets.ImageFolder(
    root=HAIR_DATASET,
    transform=train_transform
)
print("Classes:", hair_dataset.classes)
print("Number of classes:", len(hair_dataset.classes))
print("Total images:", len(hair_dataset))


Classes: ['Straight', 'Wavy', 'curly', 'dreadlocks', 'kinky']
Number of classes: 5
Total images: 1991


In [9]:
bald_dataset = datasets.ImageFolder(
    root=BALD_DATASET,
    transform=train_transform
)

print("Classes:", bald_dataset.classes)
print("Number of classes:", len(bald_dataset.classes))
print("Total images:", len(bald_dataset))


Classes: ['bald', 'notbald']
Number of classes: 2
Total images: 1113


# Split into training and validation.

In [10]:

from torch.utils.data import random_split

# --- 1. Split Hair Dataset (1,991 images) ---
hair_total = len(hair_dataset)
hair_train_len = int(0.8 * hair_total)
hair_val_len = hair_total - hair_train_len
hair_train_set, hair_val_set = random_split(hair_dataset, [hair_train_len, hair_val_len])

# --- 2. Split Bald Dataset (25 images) ---
bald_total = len(bald_dataset)
bald_train_len = int(0.8 * bald_total) # 20 images
bald_val_len = bald_total - bald_train_len   # 5 images
bald_train_set, bald_val_set = random_split(bald_dataset, [bald_train_len, bald_val_len])

print(f"Hair Dataset -> Train: {len(hair_train_set)} | Val: {len(hair_val_set)}")
print(f"Bald Dataset -> Train: {len(bald_train_set)} | Val: {len(bald_val_set)}")


Hair Dataset -> Train: 1592 | Val: 399
Bald Dataset -> Train: 890 | Val: 223


# Create dataloaders

In [11]:
BATCH_SIZE = 32
hair_train_loader =DataLoader(
    hair_train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
   
)
hair_val_loader = DataLoader(
    hair_val_set, 
    batch_size=32, 
    shuffle=False, 
    num_workers=2, 
    
)
print(f"\nHair Loaders -> {len(hair_train_loader)} train batches, {len(hair_val_loader)} val batches.")


Hair Loaders -> 50 train batches, 13 val batches.


In [12]:
bald_train_loader = DataLoader(
    bald_train_set, batch_size=4, shuffle=True, num_workers=2
)
bald_val_loader = DataLoader(
    bald_val_set, batch_size=2, shuffle=False, num_workers=2
)
print(f"Bald Loaders -> {len(bald_train_loader)} train batches, {len(bald_val_loader)} val batches.")

Bald Loaders -> 223 train batches, 112 val batches.


In [13]:
# Hair dataset
images, labels = next(iter(hair_train_loader))

print("Hair Images Shape:", images.shape)
print("Hair Labels Shape:", labels.shape)
print("Hair Labels:", labels)

Hair Images Shape: torch.Size([32, 3, 224, 224])
Hair Labels Shape: torch.Size([32])
Hair Labels: tensor([0, 1, 4, 1, 1, 0, 0, 1, 1, 2, 1, 3, 2, 3, 2, 0, 3, 3, 2, 3, 1, 1, 3, 3,
        4, 0, 0, 3, 0, 3, 2, 3])


In [14]:
# Bald dataset
images, labels = next(iter(bald_train_loader))

print("Bald Images Shape:", images.shape)
print("Bald Labels Shape:", labels.shape)
print("Bald Labels:", labels)

Bald Images Shape: torch.Size([4, 3, 224, 224])
Bald Labels Shape: torch.Size([4])
Bald Labels: tensor([1, 0, 0, 1])


# Count images per class (Most Important)

In [15]:
from collections import Counter

hair_labels = [label for _, label in hair_dataset]

print(Counter(hair_labels))
print(hair_dataset.classes)

Counter({2: 513, 0: 488, 3: 443, 1: 330, 4: 217})
['Straight', 'Wavy', 'curly', 'dreadlocks', 'kinky']


In [16]:
from collections import Counter

bald_labels = [label for _, label in bald_dataset]

print(Counter(bald_labels))
print(bald_dataset.classes)

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Counter({0: 560, 1: 553})
['bald', 'notbald']


# Check train/validation distribution

Random splitting sometimes gives one class much fewer samples.

In [17]:
from collections import Counter

train_labels = [hair_train_set[i][1] for i in range(len(hair_train_set))]
val_labels = [hair_val_set[i][1] for i in range(len(hair_val_set))]

print("Train:", Counter(train_labels))
print("Validation:", Counter(val_labels))

Train: Counter({2: 407, 0: 381, 3: 354, 1: 271, 4: 179})
Validation: Counter({0: 107, 2: 106, 3: 89, 1: 59, 4: 38})


# Visualize random images from every class

In [18]:
# import matplotlib.pyplot as plt

# for c in range(len(hair_dataset.classes)):
#     for img, label in hair_dataset:
#         if label == c:
#             img = img.permute(1,2,0).numpy()
#             img = img * [0.229,0.224,0.225] + [0.485,0.456,0.406]
#             img = img.clip(0,1)

#             plt.figure(figsize=(3,3))
#             plt.imshow(img)
#             plt.title(hair_dataset.classes[c])
#             plt.axis("off")
#             break

In [19]:


# for c in range(len(bald_dataset.classes)):
#     for img, label in bald_dataset:
#         if label == c:
#             img = img.permute(1,2,0).numpy()
#             img = img * [0.229,0.224,0.225] + [0.485,0.456,0.406]
#             img = img.clip(0,1)

#             plt.figure(figsize=(3,3))
#             plt.imshow(img)
#             plt.title(bald_dataset.classes[c])
#             plt.axis("off")
#             break

| Class      | Images |
| ---------- | -----: |
| Straight   |    488 |
| Wavy       |    330 |
| Curly      |    513 |
| Dreadlocks |    443 |
| Kinky      |    217 |

# 217 images isn't terrible
but it's about 42% of the largest class (Curly = 513). The model may learn Curly much better than Kinky.

Keep training normally

No need to throw the dataset away.

# 1. Build the CNN Architecture (nn.Sequential)

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. BUILD THE CNN MODEL
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.main_pipeline = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Block 2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Classifier Output Head
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.main_pipeline(x)

# 2. Comprehensive Master Training & Validation Function

In [21]:
# 2. THE MASTER TRAINING ENGINE FUNCTION
def run_training(model, train_loader, val_loader, save_name, epochs=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    best_accuracy = 0.0

    for epoch in range(epochs):
        # --- Train Phase ---
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # --- Validation Phase ---
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        accuracy = (correct / total) * 100
        print(f"Epoch {epoch+1} Complete | Validation Accuracy: {accuracy:.2f}%")

        # --- Save Best File ---
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            torch.save(model.state_dict(), save_name)
            print(f" Saved new top model: {save_name}")

# 3. TRIGGER TRAINING FOR BOTH DATASETS
print("--- Training Hair Type Model (5 Classes) ---")
hair_model = SimpleCNN(num_classes=5)
run_training(hair_model, hair_train_loader, hair_val_loader, "hair_type_model.pth")

print("\n--- Training Bald Classifier Model (2 Classes) ---")
bald_model = SimpleCNN(num_classes=2)
run_training(bald_model, bald_train_loader, bald_val_loader, "bald_model.pth")

--- Training Hair Type Model (5 Classes) ---
Epoch 1 Complete | Validation Accuracy: 35.84%
 Saved new top model: hair_type_model.pth
Epoch 2 Complete | Validation Accuracy: 47.12%
 Saved new top model: hair_type_model.pth
Epoch 3 Complete | Validation Accuracy: 50.63%
 Saved new top model: hair_type_model.pth
Epoch 4 Complete | Validation Accuracy: 47.37%
Epoch 5 Complete | Validation Accuracy: 46.62%

--- Training Bald Classifier Model (2 Classes) ---
Epoch 1 Complete | Validation Accuracy: 73.99%
 Saved new top model: bald_model.pth
Epoch 2 Complete | Validation Accuracy: 64.57%
Epoch 3 Complete | Validation Accuracy: 73.09%
Epoch 4 Complete | Validation Accuracy: 75.34%
 Saved new top model: bald_model.pth
Epoch 5 Complete | Validation Accuracy: 65.02%


# Replace your SimpleCNN class definition with this: